In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
storage_account_name = "ecomstorage2026"
storage_account_key = "<Your_Strorage_Account_key_here"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://landing@ecomstorage2026.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,customers.csv,1271248,1782625980000
abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,inventory.csv,576844,1782625988000
abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,order_items.csv,23501483,1782626002000
abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,orders.csv,5543233,1782626010000


In [0]:
# 1. Created databases for Medallion Architecture

spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_landing")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_gold")

DataFrame[]

In [0]:
spark.sql("SHOW DATABASES").show(truncate=False)

+--------------------+
|databaseName        |
+--------------------+
|default             |
|ecommerce_bronze    |
|ecommerce_gold      |
|ecommerce_landing   |
|ecommerce_quarantine|
|ecommerce_silver    |
|information_schema  |
+--------------------+



# Loading all source CSV files

In [0]:
base_path = "abfss://landing@ecomstorage2026.dfs.core.windows.net/"

# Customers

In [0]:
customers_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(base_path + "customers.csv")

# Inventory

In [0]:
inventory_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(base_path + "inventory.csv")

# Orders

In [0]:
orders_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(base_path + "orders.csv")

# Order Items

In [0]:
order_items_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(base_path + "order_items.csv")

# Verifying All

In [0]:
print("Customers :", customers_df.count())
print("Inventory :", inventory_df.count())
print("Orders :", orders_df.count())
print("Order Items :", order_items_df.count())

Customers : 10000
Inventory : 5000
Orders : 50150
Order Items : 200000


# Landing Layer

In [0]:
spark.sql("USE CATALOG ecommerce_databricks")
spark.sql("USE SCHEMA ecommerce_landing")

DataFrame[]

In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "abfss://landing@ecomstorage2026.dfs.core.windows.net/"

customers_landing = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(base_path + "customers.csv")
        .withColumn("landing_timestamp", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
)

In [0]:
customers_landing.printSchema()
display(customers_landing.limit(5))

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- load_ts: string (nullable = true)
 |-- landing_timestamp: timestamp (nullable = false)
 |-- source_file_name: string (nullable = false)



customer_id,first_name,last_name,email,phone,city,state,region,signup_date,is_active,load_ts,landing_timestamp,source_file_name
CUST000001,Liam,Chaudry,Udant@,8196001338,Kishanganj,Gujarat,North,2022-02-08 20:28:06,1,2025-04-17 06:00:00,2026-07-07T07:36:16.378749Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000002,Arunima,Ahuja,ckannan@example.net,+916542351161,Hapur,Punjab,South,2022-12-13 17:53:58,1,2025-04-17 06:00:00,2026-07-07T07:36:16.378749Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000003,Kritika,Brar,caleb78@example.org,4959310341,Salem,Madhya Pradesh,North,2024-11-17 05:11:07,1,2025-04-17 06:00:00,2026-07-07T07:36:16.378749Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000004,Isha,Kadakia,sudiksha52@example.com,4192832764,Anantapur,Uttarakhand,Central,2023-10-18 10:23:08,1,2025-04-17 06:00:00,2026-07-07T07:36:16.378749Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000005,Nandini,Loyal,karnikazad@example.com,+913953767242,Kishanganj,Nagaland,North,2022-05-26 13:12:42,1,2025-04-17 06:00:00,2026-07-07T07:36:16.378749Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv


# Customers:

In [0]:
customers_landing.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("customers")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+-----------------+-----------+-----------+
|database         |tableName  |isTemporary|
+-----------------+-----------+-----------+
|ecommerce_landing|customers  |false      |
|ecommerce_landing|inventory  |false      |
|ecommerce_landing|order_items|false      |
|ecommerce_landing|orders     |false      |
+-----------------+-----------+-----------+



In [0]:
display(spark.table("customers"))

customer_id,first_name,last_name,email,phone,city,state,region,signup_date,is_active,load_ts,landing_timestamp,source_file_name
CUST000001,Liam,Chaudry,Udant@,8196001338,Kishanganj,Gujarat,North,2022-02-08 20:28:06,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000002,Arunima,Ahuja,ckannan@example.net,+916542351161,Hapur,Punjab,South,2022-12-13 17:53:58,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000003,Kritika,Brar,caleb78@example.org,4959310341,Salem,Madhya Pradesh,North,2024-11-17 05:11:07,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000004,Isha,Kadakia,sudiksha52@example.com,4192832764,Anantapur,Uttarakhand,Central,2023-10-18 10:23:08,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000005,Nandini,Loyal,karnikazad@example.com,+913953767242,Kishanganj,Nagaland,North,2022-05-26 13:12:42,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000006,Theodore,Devi,priya96@example.com,7101226916,Kavali,Manipur,South,2024-02-25 01:00:38,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000007,Harini,Choudhury,advay18@example.com,06270482814,Jamnagar,Kerala,Central,2022-11-05 21:12:17,0,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000008,Anirudh,Choudhury,aadinaik@example.com,04303911718,Warangal,Sikkim,South,2023-11-29 19:43:37,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000009,Pallavi,Kakar,qgandhi@example.net,3465787133,Sonipat,Andhra Pradesh,North,2025-03-24 18:23:20,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv
CUST000010,Daniel,Karnik,thomasamble@example.org,+910518347382,Latur,Uttar Pradesh,West,2023-06-13 13:27:31,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv


# Orders:

In [0]:
orders_landing = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(base_path + "orders.csv")
        .withColumn("landing_timestamp", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
)

In [0]:
orders_landing.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("orders")

# Order Items:

In [0]:
order_items_landing = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(base_path + "order_items.csv")
        .withColumn("landing_timestamp", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
)

In [0]:
order_items_landing.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("order_items")

# Inventory:

In [0]:
inventory_landing = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(base_path + "inventory.csv")
        .withColumn("landing_timestamp", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
)

In [0]:
inventory_landing.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("inventory")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+-----------------+-----------+-----------+
|database         |tableName  |isTemporary|
+-----------------+-----------+-----------+
|ecommerce_landing|customers  |false      |
|ecommerce_landing|inventory  |false      |
|ecommerce_landing|order_items|false      |
|ecommerce_landing|orders     |false      |
+-----------------+-----------+-----------+



# Bronze Layer:

In [0]:
spark.sql("USE SCHEMA ecommerce_bronze")

DataFrame[]

In [0]:
customers_bronze = spark.table("ecommerce_databricks.ecommerce_landing.customers")
orders_bronze = spark.table("ecommerce_databricks.ecommerce_landing.orders")
order_items_bronze = spark.table("ecommerce_databricks.ecommerce_landing.order_items")
inventory_bronze = spark.table("ecommerce_databricks.ecommerce_landing.inventory")

In [0]:
from pyspark.sql.functions import current_timestamp, current_date

# Customers:

In [0]:
customers_bronze = (
    customers_bronze
        .withColumn("bronze_ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

# Orders:

In [0]:
orders_bronze = (
    orders_bronze
        .withColumn("bronze_ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

# Order Items:

In [0]:
order_items_bronze = (
    order_items_bronze
        .withColumn("bronze_ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

# Inventory:

In [0]:
inventory_bronze = (
    inventory_bronze
        .withColumn("bronze_ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

In [0]:
customers_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("load_date") \
    .saveAsTable("customers")

In [0]:
orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("load_date") \
    .saveAsTable("orders")

In [0]:
order_items_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("load_date") \
    .saveAsTable("order_items")

In [0]:
inventory_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("load_date") \
    .saveAsTable("inventory")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+----------------+-----------+-----------+
|database        |tableName  |isTemporary|
+----------------+-----------+-----------+
|ecommerce_bronze|customers  |false      |
|ecommerce_bronze|inventory  |false      |
|ecommerce_bronze|order_items|false      |
|ecommerce_bronze|orders     |false      |
+----------------+-----------+-----------+



# Silver Layer - Orders

In [0]:
spark.sql("USE CATALOG ecommerce_databricks")
spark.sql("USE SCHEMA ecommerce_silver")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
orders_bronze = spark.table(
    "ecommerce_databricks.ecommerce_bronze.orders"
)

In [0]:
orders_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- load_ts: string (nullable = true)
 |-- landing_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- load_date: date (nullable = true)



# Deduplication

In [0]:
window_spec = Window.partitionBy("order_id") \
    .orderBy(F.col("bronze_ingestion_timestamp").desc())

orders_dedup = (
    orders_bronze
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
)

# Typecasting

In [0]:
orders_cast = (
    orders_dedup
        .withColumn("order_date", F.to_timestamp("order_date"))
        .withColumn("total_amount", F.col("total_amount").cast("double"))
        .withColumn("discount_amount", F.col("discount_amount").cast("double"))
)

In [0]:
valid_status = [
    "placed",
    "shipped",
    "delivered",
    "cancelled",
    "refunded"
]

orders_quarantine = (
    orders_cast
        .filter(
            (F.col("order_id").isNull()) |
            (F.col("customer_id").isNull()) |
            (~F.lower(F.col("status")).isin([s.lower() for s in valid_status])) |
            (F.col("total_amount") <= 0)
        )
        .withColumn(
            "quarantine_reason",
            F.when(F.col("order_id").isNull(), "order_id is NULL")
             .when(F.col("customer_id").isNull(), "customer_id is NULL")
             .when(~F.lower(F.col("status")).isin([s.lower() for s in valid_status]), "Invalid Status")
             .otherwise("Invalid Total Amount")
        )
)

In [0]:
orders_silver = (
    orders_cast
        .filter(
            (F.col("order_id").isNotNull()) &
            (F.col("customer_id").isNotNull()) &
            (F.lower(F.col("status")).isin([s.lower() for s in valid_status])) &
            (F.col("total_amount") > 0)
        )
)

# Checking record counts

In [0]:
print("Total Bronze Records :", orders_bronze.count())
print("Silver Records       :", orders_silver.count())
print("Quarantine Records   :", orders_quarantine.count())

Total Bronze Records : 100300
Silver Records       : 43590
Quarantine Records   : 6410


In [0]:
spark.sql("USE CATALOG ecommerce_databricks")
spark.sql("USE SCHEMA ecommerce_silver")

DataFrame[]

In [0]:
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders")

# Creating Quarantine Schema

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS ecommerce_quarantine
""")

DataFrame[]

In [0]:
spark.sql("USE SCHEMA ecommerce_quarantine")

DataFrame[]

In [0]:
orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders_quarantine")

In [0]:
spark.sql("""
SHOW TABLES IN ecommerce_databricks.ecommerce_silver
""").show(truncate=False)

+----------------+-----------+-----------+
|database        |tableName  |isTemporary|
+----------------+-----------+-----------+
|ecommerce_silver|customers  |false      |
|ecommerce_silver|inventory  |false      |
|ecommerce_silver|order_items|false      |
|ecommerce_silver|orders     |false      |
+----------------+-----------+-----------+



In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_databricks.ecommerce_silver.orders
        LIMIT 5
    """)
)

order_id,customer_id,order_date,status,total_amount,discount_amount,payment_method,warehouse_id,region,load_ts,landing_timestamp,source_file_name,bronze_ingestion_timestamp,load_date
ORD00000002,CUST005368,2025-01-11T07:38:06Z,delivered,18686.12,5117.37,UPI,WH-BLR,South,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07
ORD00000004,CUST007879,2025-02-23T03:43:08Z,delivered,48334.85,8979.02,credit_card,WH-BLR,Central,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07
ORD00000006,CUST001519,2025-03-08T05:45:29Z,refunded,32925.05,2494.44,net_banking,WH-DEL,North,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07
ORD00000008,CUST007016,2025-03-02T21:57:33Z,delivered,18165.58,3635.19,debit_card,WH-HYD,South,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07
ORD00000009,CUST008154,2025-03-26T04:32:02Z,refunded,24864.44,2689.67,COD,WH-BLR,Central,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_databricks.ecommerce_quarantine.orders_quarantine
        LIMIT 5
    """)
)

order_id,customer_id,order_date,status,total_amount,discount_amount,payment_method,warehouse_id,region,load_ts,landing_timestamp,source_file_name,bronze_ingestion_timestamp,load_date,quarantine_reason
ORD00000001,null,2025-03-09T21:35:49Z,placed,18079.31,4993.93,debit_card,WH-CHE,North,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07,customer_id is NULL
ORD00000003,CUST008905,2025-03-06T15:59:27Z,unknown,37342.13,5086.63,credit_card,WH-CHE,North,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07,Invalid Status
ORD00000005,CUST008514,2025-01-23T03:58:49Z,PENDING,5811.5,1665.29,UPI,WH-HYD,West,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07,Invalid Status
ORD00000007,CUST000062,2025-03-12T13:45:41Z,PENDING,20392.98,2539.08,debit_card,WH-HYD,West,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07,Invalid Status
ORD00000011,CUST001676,2025-03-16T00:30:34Z,PENDING,9132.17,1175.42,UPI,WH-CHE,East,2025-04-17 06:00:00,2026-07-07T07:36:24.328186Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/orders.csv,2026-07-07T07:36:45.633365Z,2026-07-07,Invalid Status


# Silver Layer - Customers

In [0]:
customers_bronze = spark.table(
    "ecommerce_databricks.ecommerce_bronze.customers"
)

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id") \
    .orderBy(F.col("bronze_ingestion_timestamp").desc())

customers_dedup = (
    customers_bronze
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
)

In [0]:
customers_cast = (
    customers_dedup
        .withColumn("signup_date", F.to_timestamp("signup_date"))
        .withColumn("is_active", F.col("is_active").cast("int"))
)

In [0]:
customers_quarantine = (
    customers_cast
        .filter(
            (F.col("customer_id").isNull()) |
            (F.col("email").isNull()) |
            (F.col("first_name").isNull()) |
            (F.col("last_name").isNull())
        )
        .withColumn(
            "quarantine_reason",
            F.when(F.col("customer_id").isNull(), "customer_id is NULL")
             .when(F.col("email").isNull(), "email is NULL")
             .when(F.col("first_name").isNull(), "first_name is NULL")
             .otherwise("last_name is NULL")
        )
)

In [0]:
customers_silver = (
    customers_cast
        .filter(
            (F.col("customer_id").isNotNull()) &
            (F.col("email").isNotNull()) &
            (F.col("first_name").isNotNull()) &
            (F.col("last_name").isNotNull())
        )
)

In [0]:
spark.sql("USE SCHEMA ecommerce_silver")

from delta.tables import DeltaTable

customers_target = (
    "ecommerce_databricks.ecommerce_silver.customers"
)

if not spark.catalog.tableExists(customers_target):

    customers_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(customers_target)

else:

    customers_delta = DeltaTable.forName(
        spark,
        customers_target
    )

    (
        customers_delta.alias("target")
        .merge(
            customers_silver.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
print("===== CUSTOMER SCD TYPE-1 MERGE VERIFICATION =====")

print(
    "Silver Customer Count:",
    spark.table(
        "ecommerce_databricks.ecommerce_silver.customers"
    ).count()
)

display(
    spark.sql("""
        SELECT
            customer_id,
            first_name,
            last_name,
            email,
            city,
            state,
            region,
            signup_date,
            is_active
        FROM ecommerce_databricks.ecommerce_silver.customers
        ORDER BY customer_id
        LIMIT 10
    """)
)

===== CUSTOMER SCD TYPE-1 MERGE VERIFICATION =====
Silver Customer Count: 10000


customer_id,first_name,last_name,email,city,state,region,signup_date,is_active
CUST000001,Liam,Chaudry,Udant@,Kishanganj,Gujarat,North,2022-02-08T20:28:06Z,1
CUST000002,Arunima,Ahuja,ckannan@example.net,Hapur,Punjab,South,2022-12-13T17:53:58Z,1
CUST000003,Kritika,Brar,caleb78@example.org,Salem,Madhya Pradesh,North,2024-11-17T05:11:07Z,1
CUST000004,Isha,Kadakia,sudiksha52@example.com,Anantapur,Uttarakhand,Central,2023-10-18T10:23:08Z,1
CUST000005,Nandini,Loyal,karnikazad@example.com,Kishanganj,Nagaland,North,2022-05-26T13:12:42Z,1
CUST000006,Theodore,Devi,priya96@example.com,Kavali,Manipur,South,2024-02-25T01:00:38Z,1
CUST000007,Harini,Choudhury,advay18@example.com,Jamnagar,Kerala,Central,2022-11-05T21:12:17Z,0
CUST000008,Anirudh,Choudhury,aadinaik@example.com,Warangal,Sikkim,South,2023-11-29T19:43:37Z,1
CUST000009,Pallavi,Kakar,qgandhi@example.net,Sonipat,Andhra Pradesh,North,2025-03-24T18:23:20Z,1
CUST000010,Daniel,Karnik,thomasamble@example.org,Latur,Uttar Pradesh,West,2023-06-13T13:27:31Z,1


In [0]:
spark.sql("USE SCHEMA ecommerce_quarantine")

customers_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customers_quarantine")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM ecommerce_databricks.ecommerce_silver.customers
    LIMIT 5
    """)
)

customer_id,first_name,last_name,email,phone,city,state,region,signup_date,is_active,load_ts,landing_timestamp,source_file_name,bronze_ingestion_timestamp,load_date
CUST000084,William,Saini,wazir41@example.org,9711798089,Asansol,Jharkhand,North,2023-11-20T09:06:41Z,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,2026-07-07T07:36:42.875673Z,2026-07-07
CUST000129,Radha,Narula,gargfitan@example.com,1803397401,Tinsukia,Odisha,West,2022-07-17T23:30:48Z,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,2026-07-07T07:36:42.875673Z,2026-07-07
CUST000184,Gunbir,Chada,amrutabhatia@example.org,5869380686,Chapra,Haryana,Central,2024-01-23T19:05:10Z,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,2026-07-07T07:36:42.875673Z,2026-07-07
CUST000543,Neel,Bir,kmurthy@example.com,09551127474,Kolhapur,Bihar,East,2024-11-19T17:06:40Z,0,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,2026-07-07T07:36:42.875673Z,2026-07-07
CUST000589,Garima,Bhat,mannanvasatika@example.net,+919803828723,Barasat,Rajasthan,East,2022-02-26T21:19:16Z,1,2025-04-17 06:00:00,2026-07-07T07:36:18.009547Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/customers.csv,2026-07-07T07:36:42.875673Z,2026-07-07


# Silver Layer - Inventory

In [0]:
inventory_bronze = spark.table(
    "ecommerce_databricks.ecommerce_bronze.inventory"
)

In [0]:
window_spec = Window.partitionBy("sku_id") \
    .orderBy(F.col("bronze_ingestion_timestamp").desc())

inventory_dedup = (
    inventory_bronze
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
)

In [0]:
inventory_cast = (
    inventory_dedup
        .withColumn("unit_cost", F.col("unit_cost").cast("double"))
        .withColumn("stock_quantity", F.col("stock_quantity").cast("int"))
)

In [0]:
inventory_quarantine = (
    inventory_cast
        .filter(
            (F.col("sku_id").isNull()) |
            (F.col("unit_cost") < 0) |
            (F.col("stock_quantity") < 0)
        )
        .withColumn(
            "quarantine_reason",
            F.when(F.col("sku_id").isNull(), "sku_id is NULL")
             .when(F.col("unit_cost") < 0, "Negative Price")
             .otherwise("Negative Stock")
        )
)

In [0]:
inventory_silver = (
    inventory_cast
        .filter(
            (F.col("sku_id").isNotNull()) &
            (F.col("unit_cost") >= 0) &
            (F.col("stock_quantity") >= 0)
        )
)

In [0]:
spark.sql("USE SCHEMA ecommerce_silver")

inventory_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("inventory")

In [0]:
spark.sql("USE SCHEMA ecommerce_quarantine")

inventory_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("inventory_quarantine")

In [0]:
inventory_bronze.printSchema()

root
 |-- sku_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: string (nullable = true)
 |-- reorder_level: string (nullable = true)
 |-- unit_cost: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- load_ts: string (nullable = true)
 |-- landing_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- load_date: date (nullable = true)



In [0]:
display(inventory_bronze.limit(5))

sku_id,product_name,category,warehouse_id,stock_quantity,reorder_level,unit_cost,last_updated,load_ts,landing_timestamp,source_file_name,bronze_ingestion_timestamp,load_date
SKU00001,Open-architected maximized time-frame,Books,WH-MUM,null,51,9076.16,2025-04-09 07:14:39,2025-04-17 06:00:00,2026-07-07T07:36:32.990855Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,2026-07-07T07:36:51.282721Z,2026-07-07
SKU00002,Ergonomic empowering workforce,Clothing,WH-DEL,557,86,6597.28,2025-04-05 11:20:13,2025-04-17 06:00:00,2026-07-07T07:36:32.990855Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,2026-07-07T07:36:51.282721Z,2026-07-07
SKU00003,Multi-lateral dynamic utilization,Beauty,WH-DEL,1798,90,8386.89,2025-04-07 04:00:04,2025-04-17 06:00:00,2026-07-07T07:36:32.990855Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,2026-07-07T07:36:51.282721Z,2026-07-07
SKU00004,Integrated 6thgeneration frame,Grocery,WH-MUM,1839,23,13021.31,2025-04-05 10:08:15,2025-04-17 06:00:00,2026-07-07T07:36:32.990855Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,2026-07-07T07:36:51.282721Z,2026-07-07
SKU00005,Upgradable mission-critical implementation,Books,WH-DEL,76,99,6116.95,2025-04-14 03:44:16,2025-04-17 06:00:00,2026-07-07T07:36:32.990855Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/inventory.csv,2026-07-07T07:36:51.282721Z,2026-07-07


# Silver Layer - Order Items

In [0]:
order_items_bronze = spark.table(
    "ecommerce_databricks.ecommerce_bronze.order_items"
)

In [0]:
window_spec = Window.partitionBy("item_id") \
    .orderBy(F.col("bronze_ingestion_timestamp").desc())

order_items_dedup = (
    order_items_bronze
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
)

In [0]:
order_items_cast = (
    order_items_dedup
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("line_total", F.col("line_total").cast("double"))
)

In [0]:
order_items_quarantine = (
    order_items_cast
        .filter(
            (F.col("item_id").isNull()) |
            (F.col("order_id").isNull()) |
            (F.col("sku_id").isNull()) |
            (F.col("quantity") <= 0) |
            (F.col("unit_price") <= 0) |
            (F.col("line_total") <= 0)
        )
        .withColumn(
            "quarantine_reason",
            F.when(F.col("item_id").isNull(),"Item ID is NULL")
             .when(F.col("order_id").isNull(),"Order ID is NULL")
             .when(F.col("sku_id").isNull(),"SKU ID is NULL")
             .when(F.col("quantity") <=0,"Invalid Quantity")
             .when(F.col("unit_price") <=0,"Invalid Unit Price")
             .otherwise("Invalid Line Total")
        )
)

In [0]:
order_items_silver = (
    order_items_cast
        .filter(
            (F.col("item_id").isNotNull()) &
            (F.col("order_id").isNotNull()) &
            (F.col("sku_id").isNotNull()) &
            (F.col("quantity") > 0) &
            (F.col("unit_price") > 0) &
            (F.col("line_total") > 0)
        )
)

In [0]:
spark.sql("USE SCHEMA ecommerce_silver")

order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("order_items")

In [0]:
spark.sql("USE SCHEMA ecommerce_quarantine")

order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("order_items_quarantine")

In [0]:
print("Silver :", order_items_silver.count())
print("Quarantine :", order_items_quarantine.count())

Silver : 195044
Quarantine : 2000


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM ecommerce_databricks.ecommerce_silver.order_items
    LIMIT 5
    """)
)

item_id,order_id,sku_id,product_name,category,quantity,unit_price,line_total,load_ts,landing_timestamp,source_file_name,bronze_ingestion_timestamp,load_date
ITEM000000002,ORD00029814,SKU01988,Organized holistic methodology,Sports,4,10382.79,41531.16,2025-04-17 06:00:00,2026-07-07T07:36:28.427497Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,2026-07-07T07:36:48.151047Z,2026-07-07
ITEM000000003,ORD00005483,SKU02931,Business-focused attitude-oriented matrix,Grocery,3,949.86,2849.58,2025-04-17 06:00:00,2026-07-07T07:36:28.427497Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,2026-07-07T07:36:48.151047Z,2026-07-07
ITEM000000004,ORD00021103,SKU02099,Reactive analyzing time-frame,Electronics,10,11822.57,118225.7,2025-04-17 06:00:00,2026-07-07T07:36:28.427497Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,2026-07-07T07:36:48.151047Z,2026-07-07
ITEM000000005,ORD00009947,SKU02318,Enterprise-wide background hub,Clothing,6,2262.3,13573.8,2025-04-17 06:00:00,2026-07-07T07:36:28.427497Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,2026-07-07T07:36:48.151047Z,2026-07-07
ITEM000000006,ORD00011854,SKU03200,Decentralized holistic capacity,Books,3,14727.92,44183.76,2025-04-17 06:00:00,2026-07-07T07:36:28.427497Z,abfss://landing@ecomstorage2026.dfs.core.windows.net/order_items.csv,2026-07-07T07:36:48.151047Z,2026-07-07


# Gold Layer

## Daily Revenue KPI:

In [0]:
orders_silver = spark.table(
    "ecommerce_databricks.ecommerce_silver.orders"
)

order_items_silver = spark.table(
    "ecommerce_databricks.ecommerce_silver.order_items"
)

# Creating daily revenue

In [0]:
daily_revenue = (
    orders_silver.alias("o")
        .join(
            order_items_silver.alias("oi"),
            F.col("o.order_id") == F.col("oi.order_id"),
            "inner"
        )
        .withColumn(
            "order_date",
            F.to_date(F.col("o.order_date"))
        )
        .groupBy(
            "order_date",
            F.col("o.region").alias("region"),
            F.col("oi.category").alias("category")
        )
        .agg(
            F.round(
                F.sum(F.col("oi.line_total")),
                2
            ).alias("total_revenue"),

            F.countDistinct(
                F.col("o.order_id")
            ).alias("order_count"),

            F.round(
                F.sum(F.col("oi.line_total")) /
                F.countDistinct(F.col("o.order_id")),
                2
            ).alias("average_order_value")
        )
        .withColumn(
            "gold_created_timestamp",
            F.current_timestamp()
        )
)

In [0]:
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.daily_revenue"
    )

In [0]:
print("===== DAILY REVENUE KPI SUMMARY =====")

daily_revenue_summary = (
    spark.table(
        "ecommerce_databricks.ecommerce_gold.daily_revenue"
    )
    .agg(
        F.round(
            F.sum("total_revenue"),
            2
        ).alias("overall_revenue"),

        F.sum(
            "order_count"
        ).alias("aggregated_order_count"),

        F.round(
            F.avg("average_order_value"),
            2
        ).alias("average_kpi_aov"),

        F.count("*").alias("gold_row_count")
    )
)

display(daily_revenue_summary)

===== DAILY REVENUE KPI SUMMARY =====


overall_revenue,aggregated_order_count,average_kpi_aov,gold_row_count
7.07831606383E9,134713,52548.92,4240


In [0]:
display(
    spark.sql("""
        SELECT
            order_date,
            region,
            category,
            total_revenue,
            order_count,
            average_order_value,
            gold_created_timestamp
        FROM ecommerce_databricks.ecommerce_gold.daily_revenue
        ORDER BY order_date, region, category
        LIMIT 20
    """)
)

order_date,region,category,total_revenue,order_count,average_order_value,gold_created_timestamp
2025-01-01,Central,Beauty,2275009.16,44,51704.75,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Books,2658776.66,47,56569.72,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Clothing,1602321.63,36,44508.93,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Electronics,1792179.38,39,45953.32,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Grocery,2369124.3,40,59228.11,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Home & Kitchen,2045998.55,42,48714.25,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Sports,2308989.12,45,51310.87,2026-07-07T07:37:50.69417Z
2025-01-01,Central,Toys,2259291.32,40,56482.28,2026-07-07T07:37:50.69417Z
2025-01-01,East,Beauty,1467202.36,31,47329.11,2026-07-07T07:37:50.69417Z
2025-01-01,East,Books,1547587.36,31,49922.17,2026-07-07T07:37:50.69417Z


## Fulfillment KPI

In [0]:
fulfillment_kpi = (
    orders_silver
        .withColumn(
            "order_date",
            F.to_date(F.col("order_date"))
        )
        .withColumn(
            "normalized_status",
            F.lower(F.trim(F.col("status")))
        )
        .groupBy(
            "order_date",
            "warehouse_id",
            "region"
        )
        .agg(
            F.count("*").alias("total_orders"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "delivered", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("delivery_rate"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "cancelled", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("cancellation_rate"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "shipped", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("shipment_rate")
        )
        .withColumn(
            "gold_created_timestamp",
            F.current_timestamp()
        )
)

In [0]:
fulfillment_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.fulfillment_kpi"
    )

In [0]:
print("===== FULFILLMENT KPI SUMMARY =====")

fulfillment_summary = (
    orders_silver
        .withColumn(
            "normalized_status",
            F.lower(F.trim(F.col("status")))
        )
        .agg(
            F.count("*").alias("total_orders"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "delivered", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("overall_delivery_rate"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "cancelled", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("overall_cancellation_rate"),

            F.round(
                F.avg(
                    F.when(
                        F.col("normalized_status") == "shipped", 1.0
                    ).otherwise(0.0)
                ) * 100,
                2
            ).alias("overall_shipment_rate")
        )
)

display(fulfillment_summary)

===== FULFILLMENT KPI SUMMARY =====


total_orders,overall_delivery_rate,overall_cancellation_rate,overall_shipment_rate
43590,23.31,23.18,23.26


In [0]:
display(
    spark.sql("""
        SELECT
            order_date,
            warehouse_id,
            region,
            total_orders,
            delivery_rate,
            cancellation_rate,
            shipment_rate,
            gold_created_timestamp
        FROM ecommerce_databricks.ecommerce_gold.fulfillment_kpi
        ORDER BY order_date, warehouse_id, region
        LIMIT 20
    """)
)

order_date,warehouse_id,region,total_orders,delivery_rate,cancellation_rate,shipment_rate,gold_created_timestamp
2025-01-01,WH-BLR,Central,16,18.75,37.5,25.0,2026-07-07T07:37:56.770476Z
2025-01-01,WH-BLR,East,15,40.0,13.33,26.67,2026-07-07T07:37:56.770476Z
2025-01-01,WH-BLR,North,29,27.59,24.14,17.24,2026-07-07T07:37:56.770476Z
2025-01-01,WH-BLR,South,15,13.33,20.0,20.0,2026-07-07T07:37:56.770476Z
2025-01-01,WH-BLR,West,14,0.0,14.29,35.71,2026-07-07T07:37:56.770476Z
2025-01-01,WH-CHE,Central,21,23.81,28.57,14.29,2026-07-07T07:37:56.770476Z
2025-01-01,WH-CHE,East,15,46.67,13.33,13.33,2026-07-07T07:37:56.770476Z
2025-01-01,WH-CHE,North,13,38.46,30.77,7.69,2026-07-07T07:37:56.770476Z
2025-01-01,WH-CHE,South,12,25.0,25.0,25.0,2026-07-07T07:37:56.770476Z
2025-01-01,WH-CHE,West,19,26.32,26.32,5.26,2026-07-07T07:37:56.770476Z


## Inventory Health KPI:

In [0]:
inventory_silver = spark.table(
    "ecommerce_databricks.ecommerce_silver.inventory"
)

latest_order_date = (
    orders_silver
        .agg(F.max(F.to_date("order_date")).alias("latest_date"))
        .first()["latest_date"]
)

demand_30d = (
    order_items_silver.alias("oi")
        .join(
            orders_silver.alias("o"),
            F.col("oi.order_id") == F.col("o.order_id"),
            "inner"
        )
        .filter(
            F.to_date(F.col("o.order_date")) >=
            F.date_sub(F.lit(latest_order_date), 29)
        )
        .groupBy(
            F.col("oi.sku_id").alias("sku_id")
        )
        .agg(
            F.sum("oi.quantity").alias("demand_30d")
        )
)

In [0]:
inventory_health = (
    inventory_silver.alias("i")
        .join(
            demand_30d.alias("d"),
            F.col("i.sku_id") == F.col("d.sku_id"),
            "left"
        )
        .select(
            F.col("i.sku_id").alias("sku_id"),
            F.col("i.product_name").alias("product_name"),
            F.col("i.category").alias("category"),
            F.col("i.warehouse_id").alias("warehouse_id"),
            F.col("i.stock_quantity").alias("stock_quantity"),
            F.col("i.reorder_level").alias("reorder_level"),
            F.col("i.unit_cost").alias("unit_cost"),
            F.coalesce(F.col("d.demand_30d"), F.lit(0)).alias("demand_30d")
        )
        .withColumn(
            "stock_status",
            F.when(
                F.col("stock_quantity") == 0,
                "stockout"
            )
            .when(
                F.col("stock_quantity") <= F.col("reorder_level"),
                "below_reorder"
            )
            .when(
                F.col("stock_quantity") > (F.col("reorder_level") * 2),
                "overstock"
            )
            .otherwise("healthy")
        )
        .withColumn(
            "reorder_flag",
            F.when(
                F.col("stock_quantity") <= F.col("reorder_level"),
                True
            ).otherwise(False)
        )
        .withColumn(
            "inventory_value",
            F.round(
                F.col("stock_quantity") * F.col("unit_cost"),
                2
            )
        )
        .withColumn(
            "gold_created_timestamp",
            F.current_timestamp()
        )
)

In [0]:
inventory_health.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.inventory_health"
    )

In [0]:
print("===== INVENTORY HEALTH KPI SUMMARY =====")

inventory_summary = (
    spark.table(
        "ecommerce_databricks.ecommerce_gold.inventory_health"
    )
    .agg(
        F.count("*").alias("total_skus"),

        F.sum(
            F.when(F.col("stock_status") == "stockout", 1).otherwise(0)
        ).alias("stockout_count"),

        F.sum(
            F.when(F.col("stock_status") == "below_reorder", 1).otherwise(0)
        ).alias("below_reorder_count"),

        F.sum(
            F.when(F.col("stock_status") == "overstock", 1).otherwise(0)
        ).alias("overstock_count"),

        F.round(
            F.sum("inventory_value"),
            2
        ).alias("total_inventory_value")
    )
)

display(inventory_summary)

===== INVENTORY HEALTH KPI SUMMARY =====


total_skus,stockout_count,below_reorder_count,overstock_count,total_inventory_value
4848,2,144,4562,3.659732125746E10


In [0]:
display(
    spark.sql("""
        SELECT
            sku_id,
            product_name,
            category,
            warehouse_id,
            stock_quantity,
            reorder_level,
            unit_cost,
            demand_30d,
            stock_status,
            reorder_flag,
            inventory_value,
            gold_created_timestamp
        FROM ecommerce_databricks.ecommerce_gold.inventory_health
        ORDER BY sku_id
        LIMIT 20
    """)
)

sku_id,product_name,category,warehouse_id,stock_quantity,reorder_level,unit_cost,demand_30d,stock_status,reorder_flag,inventory_value,gold_created_timestamp
SKU00002,Ergonomic empowering workforce,Clothing,WH-DEL,557,86,6597.28,58,overstock,false,3674684.96,2026-07-07T07:38:04.144914Z
SKU00003,Multi-lateral dynamic utilization,Beauty,WH-DEL,1798,90,8386.89,65,overstock,false,1.507962822E7,2026-07-07T07:38:04.144914Z
SKU00004,Integrated 6thgeneration frame,Grocery,WH-MUM,1839,23,13021.31,89,overstock,false,2.394618909E7,2026-07-07T07:38:04.144914Z
SKU00005,Upgradable mission-critical implementation,Books,WH-DEL,76,99,6116.95,34,below_reorder,true,464888.2,2026-07-07T07:38:04.144914Z
SKU00006,Focused 24/7 solution,Toys,WH-BLR,1225,55,3537.4,24,overstock,false,4333315.0,2026-07-07T07:38:04.144914Z
SKU00007,Future-proofed homogeneous challenge,Clothing,WH-CHE,1481,42,12380.98,63,overstock,false,1.833623138E7,2026-07-07T07:38:04.144914Z
SKU00008,Pre-emptive non-volatile access,Books,WH-MUM,797,16,1452.55,34,overstock,false,1157682.35,2026-07-07T07:38:04.144914Z
SKU00009,Integrated human-resource solution,Books,WH-HYD,1580,86,179.72,62,overstock,false,283957.6,2026-07-07T07:38:04.144914Z
SKU00010,User-friendly system-worthy parallelism,Beauty,WH-DEL,1273,93,11638.79,41,overstock,false,1.481617967E7,2026-07-07T07:38:04.144914Z
SKU00011,Profit-focused zero administration access,Toys,WH-BLR,626,84,10451.66,50,overstock,false,6542739.16,2026-07-07T07:38:04.144914Z


## Customer LTV KPI

In [0]:
customers_silver = spark.table(
    "ecommerce_databricks.ecommerce_silver.customers"
)

latest_order_date = (
    orders_silver
        .agg(F.max(F.to_date("order_date")).alias("latest_date"))
        .first()["latest_date"]
)

customer_metrics = (
    orders_silver
        .groupBy("customer_id")
        .agg(
            F.round(
                F.sum("total_amount"),
                2
            ).alias("lifetime_spend"),

            F.countDistinct(
                "order_id"
            ).alias("order_frequency"),

            F.max(
                F.to_date("order_date")
            ).alias("last_order_date")
        )
        .withColumn(
            "recency_days",
            F.datediff(
                F.lit(latest_order_date),
                F.col("last_order_date")
            )
        )
)

# Performed left join

In [0]:
customer_ltv_base = (
    customers_silver.alias("c")
        .join(
            customer_metrics.alias("m"),
            F.col("c.customer_id") == F.col("m.customer_id"),
            "left"
        )
        .select(
            F.col("c.customer_id").alias("customer_id"),
            F.col("c.first_name").alias("first_name"),
            F.col("c.last_name").alias("last_name"),
            F.col("c.email").alias("email"),
            F.col("c.city").alias("city"),
            F.col("c.state").alias("state"),
            F.col("c.region").alias("region"),
            F.col("c.signup_date").alias("signup_date"),
            F.col("c.is_active").alias("is_active"),

            F.coalesce(
                F.col("m.lifetime_spend"),
                F.lit(0.0)
            ).alias("lifetime_spend"),

            F.coalesce(
                F.col("m.order_frequency"),
                F.lit(0)
            ).alias("order_frequency"),

            F.col("m.last_order_date").alias("last_order_date"),

            F.col("m.recency_days").alias("recency_days")
        )
)

In [0]:
ltv_thresholds = (
    customer_ltv_base
        .filter(F.col("lifetime_spend") > 0)
        .approxQuantile(
            "lifetime_spend",
            [0.30, 0.70, 0.90],
            0.01
        )
)

low_value_threshold = ltv_thresholds[0]
high_value_threshold = ltv_thresholds[1]
vip_threshold = ltv_thresholds[2]

In [0]:
customer_ltv = (
    customer_ltv_base
        .withColumn(
            "customer_segment",

            F.when(
                F.col("lifetime_spend") >= vip_threshold,
                "VIP"
            )

            .when(
                F.col("lifetime_spend") >= high_value_threshold,
                "High Value"
            )

            .when(
                F.col("lifetime_spend") >= low_value_threshold,
                "Mid Value"
            )

            .otherwise(
                "Low Value"
            )
        )
        .withColumn(
            "gold_created_timestamp",
            F.current_timestamp()
        )
)

In [0]:
customer_ltv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.customer_ltv"
    )

In [0]:
print("===== CUSTOMER LTV KPI SUMMARY =====")

customer_ltv_summary = (
    spark.table(
        "ecommerce_databricks.ecommerce_gold.customer_ltv"
    )
    .agg(
        F.count("*").alias("total_customers"),

        F.round(
            F.avg(
                F.when(
                    F.col("is_active") == 1,
                    1.0
                ).otherwise(0.0)
            ) * 100,
            2
        ).alias("active_customer_pct"),

        F.round(
            F.avg("lifetime_spend"),
            2
        ).alias("average_ltv")
    )
)

display(customer_ltv_summary)

===== CUSTOMER LTV KPI SUMMARY =====


total_customers,active_customer_pct,average_ltv
10000,74.28,109217.83


# Segment Breakdown:

In [0]:
print("===== CUSTOMER SEGMENT BREAKDOWN =====")

display(
    spark.table(
        "ecommerce_databricks.ecommerce_gold.customer_ltv"
    )
    .groupBy("customer_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(
            F.sum("lifetime_spend"),
            2
        ).alias("total_segment_ltv")
    )
    .orderBy(
        F.desc("total_segment_ltv")
    )
)

===== CUSTOMER SEGMENT BREAKDOWN =====


customer_segment,customer_count,total_segment_ltv
Mid Value,3918,4.0467387643E8
High Value,1965,3.1053321232E8
VIP,1068,2.4091884468E8
Low Value,3049,1.3605240077E8


In [0]:
display(
    spark.sql("""
        SELECT
            customer_id,
            first_name,
            last_name,
            region,
            lifetime_spend,
            order_frequency,
            last_order_date,
            recency_days,
            customer_segment,
            gold_created_timestamp
        FROM ecommerce_databricks.ecommerce_gold.customer_ltv
        ORDER BY lifetime_spend DESC
        LIMIT 20
    """)
)

customer_id,first_name,last_name,region,lifetime_spend,order_frequency,last_order_date,recency_days,customer_segment,gold_created_timestamp
CUST005332,Orinder,Kade,West,399187.57,15,2025-04-12,4,VIP,2026-07-07T07:38:12.850301Z
CUST000538,Hemang,Brahmbhatt,West,397355.52,13,2025-04-05,11,VIP,2026-07-07T07:38:12.850301Z
CUST000212,Ishaan,Chatterjee,North,385335.01,12,2025-04-04,12,VIP,2026-07-07T07:38:12.850301Z
CUST004276,Yuvraj,Batra,North,383897.36,11,2025-04-06,10,VIP,2026-07-07T07:38:12.850301Z
CUST003251,Shivansh,Parmer,East,371331.07,13,2025-04-13,3,VIP,2026-07-07T07:38:12.850301Z
CUST009186,Lopa,Nagar,Central,351275.61,11,2025-04-03,13,VIP,2026-07-07T07:38:12.850301Z
CUST004464,Girindra,Krishna,North,350356.13,12,2025-04-15,1,VIP,2026-07-07T07:38:12.850301Z
CUST008642,Qabil,Date,West,350203.87,12,2025-04-12,4,VIP,2026-07-07T07:38:12.850301Z
CUST003547,Hiral,Saha,West,347779.8,14,2025-04-14,2,VIP,2026-07-07T07:38:12.850301Z
CUST008296,Shivansh,Pandit,West,341285.6,9,2025-03-24,23,VIP,2026-07-07T07:38:12.850301Z


# Reconciliation:

In [0]:
row_count_tables = [
    ("landing", "orders", "ecommerce_databricks.ecommerce_landing.orders"),
    ("landing", "order_items", "ecommerce_databricks.ecommerce_landing.order_items"),
    ("landing", "customers", "ecommerce_databricks.ecommerce_landing.customers"),
    ("landing", "inventory", "ecommerce_databricks.ecommerce_landing.inventory"),

    ("bronze", "orders", "ecommerce_databricks.ecommerce_bronze.orders"),
    ("bronze", "order_items", "ecommerce_databricks.ecommerce_bronze.order_items"),
    ("bronze", "customers", "ecommerce_databricks.ecommerce_bronze.customers"),
    ("bronze", "inventory", "ecommerce_databricks.ecommerce_bronze.inventory"),

    ("silver", "orders", "ecommerce_databricks.ecommerce_silver.orders"),
    ("silver", "order_items", "ecommerce_databricks.ecommerce_silver.order_items"),
    ("silver", "customers", "ecommerce_databricks.ecommerce_silver.customers"),
    ("silver", "inventory", "ecommerce_databricks.ecommerce_silver.inventory"),

    ("gold", "daily_revenue", "ecommerce_databricks.ecommerce_gold.daily_revenue"),
    ("gold", "fulfillment_kpi", "ecommerce_databricks.ecommerce_gold.fulfillment_kpi"),
    ("gold", "inventory_health", "ecommerce_databricks.ecommerce_gold.inventory_health"),
    ("gold", "customer_ltv", "ecommerce_databricks.ecommerce_gold.customer_ltv")
]

row_count_records = []

for layer, table_name, full_table_name in row_count_tables:
    row_count = spark.table(full_table_name).count()

    row_count_records.append(
        (layer, table_name, full_table_name, row_count)
    )

reconciliation_row_counts = (
    spark.createDataFrame(
        row_count_records,
        ["layer", "table_name", "full_table_name", "row_count"]
    )
    .withColumn(
        "captured_at",
        F.current_timestamp()
    )
)

In [0]:
reconciliation_row_counts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.reconciliation_row_counts"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_databricks.ecommerce_gold.reconciliation_row_counts
        ORDER BY layer, table_name
    """)
)

layer,table_name,full_table_name,row_count,captured_at
bronze,customers,ecommerce_databricks.ecommerce_bronze.customers,20000,2026-07-07T07:38:24.2989Z
bronze,inventory,ecommerce_databricks.ecommerce_bronze.inventory,10000,2026-07-07T07:38:24.2989Z
bronze,order_items,ecommerce_databricks.ecommerce_bronze.order_items,400000,2026-07-07T07:38:24.2989Z
bronze,orders,ecommerce_databricks.ecommerce_bronze.orders,100300,2026-07-07T07:38:24.2989Z
gold,customer_ltv,ecommerce_databricks.ecommerce_gold.customer_ltv,10000,2026-07-07T07:38:24.2989Z
gold,daily_revenue,ecommerce_databricks.ecommerce_gold.daily_revenue,4240,2026-07-07T07:38:24.2989Z
gold,fulfillment_kpi,ecommerce_databricks.ecommerce_gold.fulfillment_kpi,2650,2026-07-07T07:38:24.2989Z
gold,inventory_health,ecommerce_databricks.ecommerce_gold.inventory_health,4848,2026-07-07T07:38:24.2989Z
landing,customers,ecommerce_databricks.ecommerce_landing.customers,20000,2026-07-07T07:38:24.2989Z
landing,inventory,ecommerce_databricks.ecommerce_landing.inventory,10000,2026-07-07T07:38:24.2989Z


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

dq_tables = [
    (
        "orders",
        "ecommerce_databricks.ecommerce_bronze.orders",
        "ecommerce_databricks.ecommerce_silver.orders",
        "ecommerce_databricks.ecommerce_quarantine.orders_quarantine"
    ),
    (
        "order_items",
        "ecommerce_databricks.ecommerce_bronze.order_items",
        "ecommerce_databricks.ecommerce_silver.order_items",
        "ecommerce_databricks.ecommerce_quarantine.order_items_quarantine"
    )
]

dq_records = []

for table_name, bronze_table, silver_table, quarantine_table in dq_tables:

    bronze_count = spark.table(bronze_table).count()
    silver_count = spark.table(silver_table).count()
    quarantine_count = spark.table(quarantine_table).count()

    pass_rate_pct = (
        float(f"{(silver_count / bronze_count) * 100:.2f}")
        if bronze_count > 0
        else 0.0
    )

    quarantine_rate_pct = (
        float(f"{(quarantine_count / bronze_count) * 100:.2f}")
        if bronze_count > 0
        else 0.0
    )

    dq_records.append(
        (
            str(table_name),
            int(bronze_count),
            int(silver_count),
            int(quarantine_count),
            pass_rate_pct,
            quarantine_rate_pct
        )
    )


dq_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("bronze_row_count", LongType(), False),
    StructField("silver_row_count", LongType(), False),
    StructField("quarantined_rows", LongType(), False),
    StructField("pass_rate_pct", DoubleType(), False),
    StructField("quarantine_rate_pct", DoubleType(), False)
])


reconciliation_dq_summary = (
    spark.createDataFrame(
        dq_records,
        schema=dq_schema
    )
    .withColumn(
        "captured_at",
        F.current_timestamp()
    )
)

In [0]:
reconciliation_dq_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ecommerce_databricks.ecommerce_gold.reconciliation_dq_summary"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_databricks.ecommerce_gold.reconciliation_dq_summary
        ORDER BY table_name
    """)
)

table_name,bronze_row_count,silver_row_count,quarantined_rows,pass_rate_pct,quarantine_rate_pct,captured_at
order_items,400000,195044,2000,48.76,0.5,2026-07-07T07:38:29.827Z
orders,100300,43590,6410,43.46,6.39,2026-07-07T07:38:29.827Z
